# RAG

In [3]:
from pypdf import PdfReader

reader = PdfReader("Motivation Letter Cassino (1).pdf")
print("Pages :", len(reader.pages))

text = "\n".join(page.extract_text() for page in reader.pages)
print("Caractères :", len(text))
print(text[:500])

Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)


Pages : 1
Caractères : 2259
Zine Eddine Tahri
Annaba, Algeria
+213542587254
tahrizinou94@gmail.com
Date: February 9, 2026
To: Admissions Committee, University of Cassino 
Subject: Motivation Letter for the Bachelor’s Degree in Computer Engineering
Dear Members of the Admissions Committee,
I am writing to formally express my strong interest in joining the Bachelor’s Degree in Computer 
Engineering at the University of Cassino. Currently in my fourth year of studies at the Higher 
School of Computer Science in Sidi Bel Abbès


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

for size in [200, 500, 1000]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=size // 5)
    chunks = splitter.split_text(text)
    print(f"chunk_size={size} -> {len(chunks)} chunks")
    print(f"  chunk[0] ({len(chunks[0])} car.) : {chunks[0][:80]!r}")
    print(f"  chunk[1] ({len(chunks[1])} car.) : {chunks[1][:80]!r}\n")

chunk_size=200 -> 14 chunks
  chunk[0] (141 car.) : 'Zine Eddine Tahri\nAnnaba, Algeria\n+213542587254\ntahrizinou94@gmail.com\nDate: Feb'
  chunk[1] (118 car.) : 'Subject: Motivation Letter for the Bachelor’s Degree in Computer Engineering\nDea'

chunk_size=500 -> 7 chunks
  chunk[0] (454 car.) : 'Zine Eddine Tahri\nAnnaba, Algeria\n+213542587254\ntahrizinou94@gmail.com\nDate: Feb'
  chunk[1] (423 car.) : 'Engineering at the University of Cassino. Currently in my fourth year of studies'

chunk_size=1000 -> 3 chunks
  chunk[0] (979 car.) : 'Zine Eddine Tahri\nAnnaba, Algeria\n+213542587254\ntahrizinou94@gmail.com\nDate: Feb'
  chunk[1] (970 car.) : 'innovation. By joining the University of Cassino, I aim to master the synergy be'



In [11]:
from sentence_transformers import SentenceTransformer

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_text(text)

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings = model.encode(chunks)

print("Nombre de chunks :", len(chunks))
print("Forme des embeddings :", embeddings.shape)
print("Premiers nombres du vecteur 0 :", embeddings[0][:5])

/Users/tahrizineeddine/Projects/rag-tutorial/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1030 20:10:11.233000 1653 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12969.12it/s]


Nombre de chunks : 7
Forme des embeddings : (7, 384)
Premiers nombres du vecteur 0 : [ 0.11630364  0.12267023 -0.05179381 -0.21355829  0.01593295]


In [12]:
import faiss
import numpy as np

vectors = np.array(embeddings, dtype="float32")
faiss.normalize_L2(vectors)

index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)

print("Vecteurs dans l'index :", index.ntotal)
print("Dimension :", index.d)

Vecteurs dans l'index : 7
Dimension : 384


In [13]:
question = "Why does the student want to join the University of Cassino?"

q_vec = model.encode([question]).astype("float32")
faiss.normalize_L2(q_vec)

scores, ids = index.search(q_vec, 3)

for rank, (score, i) in enumerate(zip(scores[0], ids[0]), start=1):
    print(f"#{rank} | chunk {i} | score = {score:.3f}")
    print(chunks[i][:300])
    print("-" * 60)

#1 | chunk 0 | score = 0.479
Zine Eddine Tahri
Annaba, Algeria
+213542587254
tahrizinou94@gmail.com
Date: February 9, 2026
To: Admissions Committee, University of Cassino 
Subject: Motivation Letter for the Bachelor’s Degree in Computer Engineering
Dear Members of the Admissions Committee,
I am writing to formally express my st
------------------------------------------------------------
#2 | chunk 5 | score = 0.456
I am fully aware of the dedication required for this program and am prepared to bring the problem-
solving mindset I developed at my school to your university. I am conﬁdent that my background in 
AI, combined with the comprehensive engineering training at Cassino, will allow me to contribute 
meani
------------------------------------------------------------
#3 | chunk 2 | score = 0.454
My current specialization has provided me with a solid foundation in how data drives modern 
innovation. By joining the University of Cassino, I aim to master the synergy between embedded 


In [14]:
%pip install ollama


[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [15]:
import ollama

context = "\n\n".join(chunks[i] for i in ids[0])

prompt = f"""Réponds à la question en utilisant UNIQUEMENT le contexte ci-dessous.
Si la réponse n'est pas dans le contexte, dis "Je ne sais pas".

Contexte :
{context}

Question : {question}
Réponse :"""

response = ollama.chat(model="llama3.2", messages=[{"role": "user", "content": prompt}])
print(response["message"]["content"])

Le étudiant souhaite rejoindre l'Université de Cassino parce qu'elle met l'accent sur les méthodes d'ingénierie pour concevoir des infrastructures informatiques sécurisées et high-performance capables de traiter de grandes quantités de données, ce qui correspond à ses intérêts en matière de data-driven intelligence.


In [18]:
def ask(question, k=3):
    q_vec = model.encode([question]).astype("float32")
    faiss.normalize_L2(q_vec)
    scores, ids = index.search(q_vec, k)
    context = "\n\n".join(chunks[i] for i in ids[0])
    prompt = f"""Réponds à la question en utilisant UNIQUEMENT le contexte ci-dessous.
Si la réponse n'est pas dans le contexte, dis "Je ne sais pas".

Contexte :
{context}

Question : {question}
Réponse :"""
    r = ollama.chat(model="llama3.2", messages=[{"role": "user", "content": prompt}])
    return r["message"]["content"]

questions = [
    "Quel est le numéro de téléphone de l'étudiant ?",
    "Dans quelle école l'étudiant étudie-t-il actuellement ?",
    "Quel est le salaire de l'étudiant ?",
    "Quelle est la capitale de l'Italie ?",
]

q = "Résume en une phrase le parcours et les motivations de l'étudiant."
print("k=1 :", ask(q, k=1))
print("-" * 60)
print("k=3 :", ask(q, k=3))

k=1 : Je ne sais pas.
------------------------------------------------------------
k=3 : Zine Eddine Tahri, étudiant actuellement en 4e année des hautes études algériennes, souhaite rejoindre la licence ingénieur informatique à l'Université de Cassino pour mettre en pratique ses compétences et son expérience développées dans le domaine de l'intelligence artificielle.
